In [ ]:
import os
import pandas as pd

df = pd.read_csv("full_dataset.csv")
df[["company", "ticker", "call_date", "v7", "v30", "n_sentences"]]

In [ ]:
row = df.iloc[0]  # American Express
txt_path = os.path.join(row["folder_path"], "TextSequence.txt")

with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

print(f"Number of lines: {len(lines)}")
for l in lines[:8]:
    print(repr(l))

In [ ]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from huggingface_hub import snapshot_download
path = snapshot_download("ProsusAI/finbert")
print(path)

In [ ]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import pandas as pd

df = pd.read_csv("full_dataset.csv")
txt_path = os.path.join(df.iloc[0]["folder_path"], "TextSequence.txt")
with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
    lines = [l.strip() for l in f if l.strip()]

from transformers import pipeline
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

sample = lines[:5]
for s, out in zip(sample, finbert(sample)):
    print(s[:70], "->", {o["label"]: round(o["score"], 3) for o in out})

In [ ]:
import time

def read_lines(folder_path):
    with open(os.path.join(folder_path, "TextSequence.txt"), "r", encoding="utf-8", errors="ignore") as f:
        return [l.strip() for l in f if l.strip()]

def score_sentences(lines, batch_size=16):
    outs = finbert(lines, batch_size=batch_size, truncation=True, max_length=128)
    s = pd.DataFrame([{o["label"]: o["score"] for o in out} for out in outs])
    s["score"] = s["positive"] - s["negative"]   # ranges from -1 to +1
    return s

def aggregate_sentiment(s):
    half = len(s) // 2
    first, second = s["score"].iloc[:half].mean(), s["score"].iloc[half:].mean()
    return {
        "sent_mean": s["score"].mean(),
        "sent_std": s["score"].std(),
        "sent_neg_share": (s["negative"] > s["positive"]).mean(),
        "sent_first_half": first,
        "sent_second_half": second,
        "sent_trend": second - first,
    }

t = time.time()
amex_lines = read_lines(df.iloc[0]["folder_path"])
amex_scores = score_sentences(amex_lines)
print(f"{len(amex_lines)} sentences scored in {time.time()-t:.1f}s")
print(aggregate_sentiment(amex_scores))

amex_scores["text"] = amex_lines
amex_scores.nsmallest(3, "score")[["text", "positive", "negative"]]

In [ ]:
pd.set_option("display.max_colwidth", None)
amex_scores.nsmallest(3, "score")[["text", "negative"]]

In [ ]:
sent_rows = []

for idx, row in df.iterrows():
    t = time.time()
    lines = read_lines(row["folder_path"])
    feats = aggregate_sentiment(score_sentences(lines))
    feats.update({"ticker": row["ticker"], "n_text_lines": len(lines)})
    sent_rows.append(feats)
    pd.DataFrame(sent_rows).to_csv("sentiment_features.csv", index=False)  # saved after every company
    print(f"{idx+1}/18 {row['company']}: {len(lines)} lines, {time.time()-t:.0f}s")

In [ ]:
sent = pd.read_csv("sentiment_features.csv")
full = df.merge(sent, on="ticker", how="left")
full.to_csv("full_dataset_v2.csv", index=False)   # new name, so the original stays untouched

feat_cols = ["pitch_mean_avg", "pitch_mean_std", "pitch_std_avg",
             "energy_mean_avg", "energy_mean_std", "zcr_mean_avg",
             "sent_mean", "sent_std", "sent_neg_share", "sent_trend"]
targets = ["v7", "v15", "v30"]

corr = full[feat_cols + targets].corr(method="spearman").loc[feat_cols, targets].round(2)
corr

In [ ]:
sub = full[full.ticker != "GS"]
corr_noGS = sub[feat_cols + targets].corr(method="spearman").loc[feat_cols, targets].round(2)
(corr - corr_noGS).abs().max()   # largest change per target after dropping GS

In [ ]:
(corr - corr_noGS)["v30"].abs().sort_values(ascending=False).head(3)

In [ ]:
print("with GS:   ", corr.loc["energy_mean_avg"].round(2).to_dict())
print("without GS:", corr_noGS.loc["energy_mean_avg"].round(2).to_dict())

In [ ]:
from scipy.stats import spearmanr

sub = full[full.ticker != "GS"]
res = []
for t in sub.ticker:
    s = sub[sub.ticker != t]
    res.append((t, round(spearmanr(s["energy_mean_avg"], s["v30"])[0], 2)))
res = pd.DataFrame(res, columns=["dropped", "rho"]).sort_values("rho")
print(res.head(3)); print(res.tail(3))

In [ ]:
from scipy.stats import spearmanr

sub = full[full.ticker != "GS"]
print("energy vs zcr:", round(spearmanr(sub["energy_mean_avg"], sub["zcr_mean_avg"])[0], 2))
print("zcr vs v30:   ", round(spearmanr(sub["zcr_mean_avg"], sub["v30"])[0], 2))

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

def get_prices(ticker, call_date, days_before=60, days_after=40):
    d = pd.Timestamp(call_date)
    data = yf.download(ticker, start=d - pd.Timedelta(days=days_before),
                       end=d + pd.Timedelta(days=days_after), progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    return data

def vol(closes):
    r = closes.pct_change().dropna()
    if len(r) < 2:
        return np.nan
    return np.log(np.sqrt(((r - r.mean()) ** 2).sum() / len(r)))

rows = []
for _, row in df.iterrows():
    d = pd.Timestamp(row["call_date"])
    p = get_prices(row["ticker"], row["call_date"])["Close"]
    out = {"ticker": row["ticker"]}
    for n in (7, 15, 30):
        out[f"past_v{n}"] = vol(p.loc[d - pd.Timedelta(days=n):d])
    rows.append(out)

past = pd.DataFrame(rows)
chk = df.merge(past, on="ticker")
for n in (7, 15, 30):
    print(f"v{n} vs past_v{n}:", round(chk[f"v{n}"].corr(chk[f"past_v{n}"], method="spearman"), 2))